<a href="https://colab.research.google.com/github/Us9r/association-rule-mining/blob/main/association_rule_rsud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import dan Load Dataset

In [1]:
import pandas as pd
url = 'https://raw.githubusercontent.com/Us9r/association-rule-mining/main/dataset_v2/Data%20RSUD%20Sekayu%20Revisi.csv'
df = pd.read_csv(url, sep=';')

In [2]:
# Menghapus baris di mana isi 'Registration No' adalah kata 'Registration No'
df_clean = df[df['Registration No'] != 'Registration No'].copy()

#lihat cuplikan
df_clean.head()

,Registration No,MRN,Subjective,Objective,Assessment/ Diagnosis,Planning,Diagnosis Name,Unnamed: 7,Unnamed: 8
1,REG/0063,428660,Keluhan utama: gatal seluruh badan,Keadaan umum: Sakit ringan,Anemia hipokrom mikrositer,"evaluasi pengobatan Hb 7,9 -> rencana transfus...","D64.9 - Anaemia, unspecified",NaN,NaN
2,REG/0677,225601,"Keluhan utama: lemas seluruh badan, nyeri pada...",Keadaan umum: Sakit ringan,Diagnisa banding: hipokalemia DM tipe 2,evaluasi pengobatan,E10.9 - Insulin-dependent diabetes militus wit...,NaN,NaN
3,REG/0024,159401,Keluhan utama: pusing berkurang,Keadaan umum: Sakit ringan,HHD DM tipe 2,Evaluasi td dan bss,E10.9 - Insulin-dependent diabetes militus wit...,NaN,NaN
4,REG/0036,272790,"Keluhan utama: kaki sembap, BAK berbusa. Riway...",Keadaan umum: Sakit ringan,Sindrom nefrotik ICD 10.03 - Secondary Diagnos...,evaluasi lab,"N04.9 - Nephrotic syndrome, unspecified",NaN,NaN
5,REG/0037,370576,"Keluhan utama: batuk, Riwayat penyakit dahulu:...",Keadaan umum: Sakit ringan,Asma terkontrol derajat ringan ICD 10: 01 - Ma...,evaluasi faktor pencetus,"J45.9 - Asthma, unspecified",NaN,NaN


##Memilih fitur yang akan digunakan
(Feature selection)

*   Periksa nilai fitur Unnamed: 7 dan 8
*   Tidak ada nilai unique (tidak ada isi) di fitur 7 dan 8
*   Drop fitur yang tidak digunakan


In [ ]:
print("Nilai unik Unnamed: 7:", df['Unnamed: 7'].unique())
print("Nilai unik Unnamed: 8:", df['Unnamed: 8'].unique())

Nilai unik Unnamed: 7: [nan]
Nilai unik Unnamed: 8: [nan]


In [3]:
# Menghapus kolom 'Subjective', 'Objective', 'Assessment/ Diagnosis', dan 'Planning'
df_clean = df_clean.drop(columns=['Subjective', 'Objective', 'Assessment/ Diagnosis', 'Planning','Unnamed: 7', 'Unnamed: 8'])

# Menampilkan beberapa baris pertama setelah kolom dihapus
df_clean.head(3)

,Registration No,MRN,Diagnosis Name
1,REG/0063,428660,"D64.9 - Anaemia, unspecified"
2,REG/0677,225601,E10.9 - Insulin-dependent diabetes militus wit...
3,REG/0024,159401,E10.9 - Insulin-dependent diabetes militus wit...


# Data Cleaning

## Duplicates Values
Identifikasi fitur yang memiliki nilai duplikat berdasarkan

In [8]:
# Filter duplikat MRN lalu urutkan berdasarkan kolom MRN
duplicate_reg = df_clean[df_clean.duplicated(subset=['Registration No'], keep=False)].sort_values(by='Registration No')

In [9]:
duplicate_reg

,Registration No,MRN,Diagnosis Name
4,REG/0036,272790,"N04.9 - Nephrotic syndrome, unspecified"
230,REG/0036,272790,I11.9 - Hypertensive heart disease without (co...
231,REG/0037,244537,E10.9 - Insulin-dependent diabetes mellitus wi...
5,REG/0037,370576,"J45.9 - Asthma, unspecified"
235,REG/0058,316190,E10.9 - Insulin-dependent diabetes mellitus wi...
...,...,...,...
279,REG/0698,443969,K21.0 - Gastro-oesophageal reflux disease with...
225,REG/0733,205293,"E04.9 - Nontoxic goitre, unspecified"
185,REG/0733,322250,"K80.8 - Other cholelithiasis, I11.9 - Hyperten..."
255,REG/0767,360498,"E04.1 - Nontoxic single thyroid nodule, R59.9 ..."


### 96 baris duplikat berdasarkan Registration No terdeteksi

hal ini wajar terjadi karena pencatatan no registrasi reset berulang di awal atau akhir bulan

Lanjut identifikasi duplikat berdasarkan MRN

In [4]:
# Filter duplikat MRN lalu urutkan berdasarkan kolom MRN
duplicate_mrn = df_clean[df_clean.duplicated(subset=['MRN'], keep=False)].sort_values(by='MRN')

# Tampilkan seluruh baris tanpa dipotong/dibatasi tambahkan parameter
# pd.set_option('display.max_rows', None)
duplicate_mrn

,Registration No,MRN,Diagnosis Name
242,REG/0124,115306,"E05.9 - Thyrotoxicosis, unspecified"
26,REG/0157,115306,E10.5 - Insulin-dependent diabetes mellitus wi...
311,REG/0217,127380,E11.9 - Non-insulin-dependent diabetes mellitu...
61,REG/0492,127380,"I10 - Essential (primary) hypertension, R42 - ..."
342,REG/0713,127560,"E05.9 - Thyrotoxicosis, unspecified"
...,...,...,...
298,REG/0067,428660,"J45.9 - Asthma, unspecified, E11.9 - Non-insul..."
56,REG/0436,438491,E10.9 - Insulin-dependent diabetes mellitus wi...
344,REG/0446,438491,E11.9 - Non-insulin-dependent diabetes mellitu...
81,REG/0117,443697,I11.9 - Hypertensive heart disease without (co...


### 62 baris duplikat berdasarkan MRN terdeteksi
hal ini bisa terjadi karena
* Pasien datang berobat rutin
* Pasien datang dihari berbeda dengan gejala berbeda (2x)

data duplikat kembali diperiksa secara manual terlebih dahulu

In [5]:
# Mengekstrak dataframe duplicate_mrn ke file Excel
duplicate_mrn.to_excel('duplicate_mrn.xlsx', index=False)
print("File 'duplicate_mrn.xlsx' berhasil disimpan!")

File 'duplicate_mrn.xlsx' berhasil disimpan!


### Solusi Penanganan Duplikat MRN
Dalam analisis aturan asosiasi (*Association Rule Mining*), duplikasi `MRN` dengan nomor registrasi yang berbeda menunjukkan pasien yang melakukan kunjungan berulang.

Berikut adalah dua cara untuk menangani kondisi ini:

#### Opsi 1: Menggabungkan Diagnosis Berdasarkan MRN (Pendekatan Riwayat Pasien)
Semua diagnosis dari pasien (`MRN`) yang sama digabungkan menjadi satu baris transaksi tunggal.

In [ ]:
# Opsi 1: Mengelompokkan berdasarkan MRN dan menggabungkan semua Diagnosis
df_by_patient = df_clean.groupby('MRN')['Diagnosis Name'].apply(lambda x: ', '.join(x.dropna().unique())).reset_index()

print("Ukuran data setelah penggabungan berdasarkan MRN:", df_by_patient.shape)
display(df_by_patient.head())

#### Opsi 2: Menghapus Duplikat secara keseluruhan
Jika setiap kunjungan (`Registration No`) adalah transaksi mandiri yang unik, kita tidak perlu menghapus duplikat `MRN` karena itu merepresentasikan kunjungan di waktu yang berbeda.